# Smart Packing Agent Demo Notebook

Welcome to the **Smart Packing Agent** demonstration notebook. This agent goes through a complete planning, observing, and acting loop. It is not just a simple chatbot; instead, it:
1. **Plans & Receives Intent**: Takes destination, trip type, and duration.
2. **Uses a Tool**: Calls `check_weather(destination)` to get weather database info.
3. **Decides**: Uses LLM reasoning (or a local rule-based system as backup) to generate a customized list based on weather and trip type.
4. **Uses a Tool**: Calls `add_item(item)` for each item to log it in the packing list.
5. **Checks Safety (Group-of-3 Feature)**: Compares items with mandatory templates via `forgot_something(trip_type, items)` and appends missing elements.
6. **Consults Memory**: Inspects state stored inside the `Memory` class and outputs the final list.

---

## 1. Setup and Initialization
We start by importing our agent from the `src` directory and creating an instance.

In [1]:
import os
import sys

# Ensure the src/ folder is in the system path
sys.path.append(os.path.abspath('..'))

from src.agent import PackingAgent

agent = PackingAgent()

[Info] No GITHUB_TOKEN configured or placeholder detected. Running Agent in deterministic fallback mode.


## Demo 1: Cold Destination (Manali)
- **Destination**: Manali (Weather: Cold, Temperature: 5°C, Raining)
- **Trip Type**: Trekking
- **Duration**: 4 Days

We expect the agent to pack warm clothing (jacket, warm clothes), rain protection (umbrella, raincoat), trekking gear (shoes, water bottle), and log everything in its Memory.

In [2]:
print("--- RUNNING DEMO 1: MANALI ---")
result_manali = agent.run(
    destination="Manali",
    trip_type="trekking",
    duration=4
)

--- RUNNING DEMO 1: MANALI ---

=== STEP 1: Understanding User Goal & Saving to Memory ===
Destination: Manali
Trip type: trekking
Duration: 4 days

=== STEP 2: Invoking Tool: check_weather() ===
Tool check_weather('Manali') returned: {'temperature': 5, 'condition': 'cold', 'rain': True}

=== STEP 3: Deciding Packing List (Reasoning & LLM Prompting) ===
[Local Logic] Computing decisions locally using deterministic rules...
[Local Logic] Decided these items: ['Jacket', 'Warm clothes', 'Raincoat', 'Umbrella', 'Trekking shoes', 'Water bottle']

=== STEP 4: Invoking Tool: add_item() for Decided Items ===
Called add_item('Jacket') -> Tool response: {'item': 'Jacket', 'packing_list': ['Jacket']}
Called add_item('Warm clothes') -> Tool response: {'item': 'Warm clothes', 'packing_list': ['Jacket', 'Warm clothes']}
Called add_item('Raincoat') -> Tool response: {'item': 'Raincoat', 'packing_list': ['Jacket', 'Warm clothes', 'Raincoat']}
Called add_item('Umbrella') -> Tool response: {'item': 'Umb

## Demo 2: Hot Destination (Goa)
- **Destination**: Goa (Weather: Hot, Temperature: 30°C, Clear)
- **Trip Type**: Beach
- **Duration**: 3 Days

Here, the weather changes completely. The agent should adapt by recommending light clothing (T-shirts, shorts), swimwear, sunglasses, and sunscreen.

In [3]:
print("--- RUNNING DEMO 2: GOA ---")
result_goa = agent.run(
    destination="Goa",
    trip_type="beach",
    duration=3
)

--- RUNNING DEMO 2: GOA ---

=== STEP 1: Understanding User Goal & Saving to Memory ===
Destination: Goa
Trip type: beach
Duration: 3 days

=== STEP 2: Invoking Tool: check_weather() ===
Tool check_weather('Goa') returned: {'temperature': 30, 'condition': 'hot', 'rain': False}

=== STEP 3: Deciding Packing List (Reasoning & LLM Prompting) ===
[Local Logic] Computing decisions locally using deterministic rules...
[Local Logic] Decided these items: ['T-shirts', 'Shorts', 'Swimwear', 'Sunglasses', 'Sunscreen']

=== STEP 4: Invoking Tool: add_item() for Decided Items ===
Called add_item('T-shirts') -> Tool response: {'item': 'T-shirts', 'packing_list': ['T-shirts']}
Called add_item('Shorts') -> Tool response: {'item': 'Shorts', 'packing_list': ['T-shirts', 'Shorts']}
Called add_item('Swimwear') -> Tool response: {'item': 'Swimwear', 'packing_list': ['T-shirts', 'Shorts', 'Swimwear']}
Called add_item('Sunglasses') -> Tool response: {'item': 'Sunglasses', 'packing_list': ['T-shirts', 'Shorts

## Demo 3: Rainy & Long Trip (Shimla)
- **Destination**: Shimla (Weather: Cold, Temperature: 8°C, Raining)
- **Trip Type**: Trekking
- **Duration**: 5 Days (Should trigger the "extra clothes" duration rule)

The agent should include cold/rain items, trekking gear, and notice the trip is 5 days long, automatically adding "Extra clothes" to memory.

In [4]:
print("--- RUNNING DEMO 3: SHIMLA ---")
result_shimla = agent.run(
    destination="Shimla",
    trip_type="trekking",
    duration=5
)

--- RUNNING DEMO 3: SHIMLA ---

=== STEP 1: Understanding User Goal & Saving to Memory ===
Destination: Shimla
Trip type: trekking
Duration: 5 days

=== STEP 2: Invoking Tool: check_weather() ===
Tool check_weather('Shimla') returned: {'temperature': 8, 'condition': 'cold', 'rain': True}

=== STEP 3: Deciding Packing List (Reasoning & LLM Prompting) ===
[Local Logic] Computing decisions locally using deterministic rules...
[Local Logic] Decided these items: ['Jacket', 'Warm clothes', 'Raincoat', 'Umbrella', 'Trekking shoes', 'Water bottle', 'Extra clothes']

=== STEP 4: Invoking Tool: add_item() for Decided Items ===
Called add_item('Jacket') -> Tool response: {'item': 'Jacket', 'packing_list': ['Jacket']}
Called add_item('Warm clothes') -> Tool response: {'item': 'Warm clothes', 'packing_list': ['Jacket', 'Warm clothes']}
Called add_item('Raincoat') -> Tool response: {'item': 'Raincoat', 'packing_list': ['Jacket', 'Warm clothes', 'Raincoat']}
Called add_item('Umbrella') -> Tool respon

## 2. Safety Check Demonstration (Group-of-3 Feature)
We can run a quick direct test of the safety checks function `forgot_something()` to show how it flags omitted items for Trekking, Beach, or Business trips.

In [5]:
from src.tools import forgot_something

packed_items = ["Swimwear", "Sunglasses"]
missing_beach = forgot_something("beach", packed_items)

print(f"For a beach trip with packed list {packed_items}, safety check flagged as missing: {missing_beach}")

For a beach trip with packed list ['Swimwear', 'Sunglasses'], safety check flagged as missing: ['Sunscreen']
